# Notebook 2.0 — Data Cleaning & Temporal Split
**Pipeline:** Raw Ingestion -> Null Audit -> Flash Wallet Pruning -> Temporal Split -> Persistence
**Input :** data/raw/dataset.parquet  (442961 rows x 78 cols)
**Outputs:** data/processed/train_clean.parquet  |  data/processed/test_clean.parquet

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
from src.preprocessing import (
    load_raw_data,
    audit_nulls,
    apply_median_imputation,
    prune_flash_wallets,
    temporal_split,
    save_parquet,
)

RAW_PATH       = PROJECT_ROOT / "data" / "raw" / "dataset.parquet"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH     = PROCESSED_DIR / "train_clean.parquet"
TEST_PATH      = PROCESSED_DIR / "test_clean.parquet"
CUTOFF_TS      = 1_672_531_200   # 2023-01-01 00:00:00 UTC
TIMESTAMP_COL  = "borrow_timestamp"

print(f"Environment ready. Project root: {PROJECT_ROOT}")

Environment ready. Project root: /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment


In [2]:
df_raw = load_raw_data(str(RAW_PATH))

import collections
dtype_groups = collections.Counter(str(dt) for dt in df_raw.dtypes)

print("--- RAW DATA SUMMARY ---")
print(f"Shape      : {df_raw.height} rows x {df_raw.width} cols")
print(f"Memory     : {df_raw.estimated_size('mb'):.2f} MB")
print("Dtypes     :")
for dtype_name, count in dtype_groups.items():
    print(f"  {dtype_name:<8} : {count} columns")

df_raw.head(3)

File path loaded: /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/raw/dataset.parquet
Shape: 442961 x 78
Memory usage: 277.97 MB
--- RAW DATA SUMMARY ---
Shape      : 442961 rows x 78 cols
Memory     : 277.97 MB
Dtypes     :
  Int64    : 17 columns
  Float64  : 60 columns
  String   : 1 columns


borrow_block_number,borrow_timestamp,wallet_address,first_tx_timestamp,last_tx_timestamp,wallet_age,incoming_tx_count,outgoing_tx_count,net_incoming_tx_count,total_gas_paid_eth,avg_gas_paid_per_tx_eth,risky_tx_count,risky_unique_contract_count,risky_first_tx_timestamp,risky_last_tx_timestamp,risky_first_last_tx_timestamp_diff,risky_sum_outgoing_amount_eth,outgoing_tx_sum_eth,incoming_tx_sum_eth,outgoing_tx_avg_eth,incoming_tx_avg_eth,max_eth_ever,min_eth_ever,total_balance_eth,risk_factor,total_collateral_eth,total_collateral_avg_eth,total_available_borrows_eth,total_available_borrows_avg_eth,avg_weighted_risk_factor,risk_factor_above_threshold_daily_count,avg_risk_factor,max_risk_factor,borrow_amount_sum_eth,borrow_amount_avg_eth,borrow_count,repay_amount_sum_eth,…,deposit_amount_sum_eth,time_since_first_deposit,withdraw_amount_sum_eth,withdraw_deposit_diff_if_positive_eth,liquidation_count,time_since_last_liquidated,liquidation_amount_sum_eth,market_adx,market_adxr,market_apo,market_aroonosc,market_aroonup,market_atr,market_cci,market_cmo,market_correl,market_dx,market_fastk,market_fastd,market_ht_trendmode,market_linearreg_slope,market_macd_macdext,market_macd_macdfix,market_macd,market_macdsignal_macdext,market_macdsignal_macdfix,market_macdsignal,market_max_drawdown_365d,market_natr,market_plus_di,market_plus_dm,market_ppo,market_rocp,market_rocr,unique_borrow_protocol_count,unique_lending_protocol_count,target
i64,f64,str,f64,f64,f64,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,…,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64
7711117,1.5572e9,"""0x502cb8985b2c92a8d4bf309cdaa8…",1.5372e9,1.5572e9,1.9973049e7,199,438,-239,0.397391,0.000981,0,0,999999999,999999999,0,0.0,975.686105,958.353127,1.174111,1.153253,61.680231,0.060948,58.317987,0.000001,44.479139,0.0,31.57527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,…,44.410991,4026.0,0.0,0.0,0,9.99999999e8,0.0,22.357432,31.489909,-3.914837,67.777778,7.142857,9.047057,111.965749,19.928594,0.878363,31.02392,80.653832,57.459322,1,0.423859,-3.914837,1.326453,1.406507,-3.468973,1.163425,1.212219,-0.895911,4.479356,33.216622,33.415526,-2.370346,0.104294,1.104294,0,1,1
7711123,1.5572e9,"""0x502cb8985b2c92a8d4bf309cdaa8…",1.5372e9,1.5572e9,1.9973188e7,200,439,-239,0.399063,0.000983,0,0,999999999,999999999,0,0.0,975.686105,958.510238,1.171292,1.150673,61.680231,0.060948,58.317987,0.004953,44.479139,44.479139,31.418863,31.57527,0.0,0.0,0.000001,0.000001,0.15711,0.15711,1,0.0,…,44.410991,4165.0,0.0,0.0,0,9.99999999e8,0.0,22.357432,31.489909,-3.914837,67.777778,7.142857,9.047057,111.965749,19.928594,0.878363,31.02392,80.653832,57.459322,1,0.423859,-3.914837,1.326453,1.406507,-3.468973,1.163425,1.212219,-0.895911,4.479356,33.216622,33.415526,-2.370346,0.104294,1.104294,1,1,1
7711126,1.5572e9,"""0x502cb8985b2c92a8d4bf309cdaa8…",1.5372e9,1.5572e9,1.9973238e7,201,440,-239,0.400895,0.000985,0,0,999999999,999999999,0,0.0,975.686105,958.896402,1.168486,1.148379,61.680231,0.060948,58.317987,0.017103,44.479139,44.479139,31.035236,31.497067,0.000387,0.0,0.002477,0.004953,0.543275,0.271637,2,0.0,…,44.410991,4215.0,0.0,0.0,0,9.99999999e8,0.0,22.357432,31.489909,-3.914837,67.777778,7.142857,9.047057,111.965749,19.928594,0.878363,31.02392,80.653832,57.459322,1,0.423859,-3.914837,1.326453,1.406507,-3.468973,1.163425,1.212219,-0.895911,4.479356,33.216622,33.415526,-2.370346,0.104294,1.104294,1,1,1


In [3]:
null_report = audit_nulls(df_raw)

if len(null_report) == 0:
    print("NULL GATE: Data is fully clean. Imputation skipped.")
    df_audited = df_raw
else:
    print(f"NULL GATE: {len(null_report)} columns require imputation.")
    df_audited = apply_median_imputation(df_raw, list(null_report.keys()))
    
    import matplotlib.pyplot as plt
    
    null_df = pl.DataFrame({
        "column": list(null_report.keys()), 
        "null_count": list(null_report.values())
    }).to_pandas()
    
    plt.figure(figsize=(8, len(null_report) * 0.4 + 1))
    plt.barh(null_df["column"], null_df["null_count"], color="steelblue")
    plt.xlabel("Null Count")
    plt.title("Columns Requiring Imputation")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

print(f"Audited DataFrame shape: {df_audited.shape}")

Total columns scanned: 78
NULL AUDIT PASSED: 0 null values found across all columns.
NULL GATE: Data is fully clean. Imputation skipped.
Audited DataFrame shape: (442961, 78)


In [4]:
df_pruned, n_removed = prune_flash_wallets(df_audited)

print("--- PRUNING REPORT ---")
print(f"Rows removed : {n_removed}")
print(f"Rows retained: {len(df_pruned)}")
print("Target distribution after pruning:")
if "target" in df_pruned.columns:
    print(df_pruned["target"].value_counts().sort("target"))
else:
    print("No target column found.")

assert len(df_pruned) + n_removed == len(df_audited), "Row count mismatch after pruning — data loss detected!"

print("Pruning complete. DataFrame passed to temporal split.")

FLASH WALLET PRUNING:
  Rows before : 442961
  Rows removed: 0 (0.00%)
  Rows after  : 442961
--- PRUNING REPORT ---
Rows removed : 0
Rows retained: 442961
Target distribution after pruning:
shape: (2, 2)
┌────────┬────────┐
│ target ┆ count  │
│ ---    ┆ ---    │
│ i64    ┆ u32    │
╞════════╪════════╡
│ 0      ┆ 276761 │
│ 1      ┆ 166200 │
└────────┴────────┘
Pruning complete. DataFrame passed to temporal split.


In [5]:
train_df, test_df = temporal_split(df_pruned, TIMESTAMP_COL, CUTOFF_TS)

print("--- SPLIT SHAPES ---")
print(f"train_df: {train_df.shape}")
print(f"test_df : {test_df.shape}")

assert train_df[TIMESTAMP_COL].max() < CUTOFF_TS, "Temporal leak: train set contains post-cutoff rows!"
assert test_df[TIMESTAMP_COL].min() >= CUTOFF_TS, "Temporal leak: test set contains pre-cutoff rows!"

print("Temporal integrity assertions passed.")

TEMPORAL SPLIT at cutoff = 1672531200 (2023-01-01 UTC):
  Train (pre-2023) : 402754 rows  (90.9%)
  Test  (post-2023): 40207 rows  (9.1%)
  Target distribution in train -> 0: 247352  1: 155402  ratio: 0.63
  Target distribution in test  -> 0: 29409  1: 10798  ratio: 0.37
--- SPLIT SHAPES ---
train_df: (402754, 78)
test_df : (40207, 78)
Temporal integrity assertions passed.


In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
save_parquet(train_df, str(TRAIN_PATH))
save_parquet(test_df, str(TEST_PATH))

train_reload = pl.read_parquet(str(TRAIN_PATH))
test_reload  = pl.read_parquet(str(TEST_PATH))

# 1. Row count preservation
assert len(train_reload) == len(train_df), f"Train parquet corrupted: expected {len(train_df)}, got {len(train_reload)}"
assert len(test_reload) == len(test_df), f"Test parquet corrupted: expected {len(test_df)}, got {len(test_reload)}"

# 2. Column schema preservation
assert train_reload.columns == train_df.columns, "Train parquet column schema mismatch after reload!"
assert test_reload.columns == test_df.columns, "Test parquet column schema mismatch after reload!"

# 3. Total row conservation (train + test = full post-prune dataset)
assert len(train_reload) + len(test_reload) == len(df_pruned), f"Data lineage broken: {len(train_reload)} + {len(test_reload)} != {len(df_pruned)}"

# 4. No data leakage across the temporal boundary
assert train_reload[TIMESTAMP_COL].max() < CUTOFF_TS, "DATA LEAKAGE DETECTED: train_clean.parquet has post-2023 rows!"
assert test_reload[TIMESTAMP_COL].min() >= CUTOFF_TS, "DATA LEAKAGE DETECTED: test_clean.parquet has pre-2023 rows!"

# 5. Target column still present and binary
assert "target" in train_reload.columns, "Target column missing in train!"
assert set(train_reload["target"].unique().to_list()).issubset({0, 1}), "Target column contains unexpected non-binary values in train!"

print("========================================")
print("ALL ASSERTION GATES PASSED")
print("Pipeline complete.")
print(f"Train : {TRAIN_PATH}  |  {len(train_reload)} rows")
print(f"Test  : {TEST_PATH}  |  {len(test_reload)} rows")
print("========================================")

SAVED: /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/processed/train_clean.parquet  |  rows=402754  cols=78  size=93.22 MB
SAVED: /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/processed/test_clean.parquet  |  rows=40207  cols=78  size=9.35 MB
ALL ASSERTION GATES PASSED
Pipeline complete.
Train : /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/processed/train_clean.parquet  |  402754 rows
Test  : /Users/danganh71/Developer/python/Machine learning I/Project/risk_assessment/data/processed/test_clean.parquet  |  40207 rows
